# NER and Entity Resolution - spaCy + EntityRuler
This work is a part of the knowledge graph pipeline built for 26 Spring BigCo Studio at Cornell Tech, and belongs to Team 10.

The whole process of this part is run on Google Colab.

# 1. Set up GDrive

In [16]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [17]:
%cd /content/gdrive/MyDrive/NER_EntityResolution

/content/gdrive/MyDrive/NER_EntityResolution


# 2. Install required packages

In [18]:
!pip install -q pandas spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 74.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


# 3. Preprocess data

In [19]:
!python src/preprocess.py \
  --input data/mock/mock2_input.csv \
  --cleaned-output data/processed/cleaned_input_mock2.csv \
  --invalid-output data/processed/invalid_rows_mock2.csv

[preprocess] Wrote 53 valid row(s) to: data/processed/cleaned_input_mock2.csv
[preprocess] Wrote 7 invalid row(s) to: data/processed/invalid_rows_mock2.csv


In [20]:
!ls data/processed/

cleaned_input_mock1.csv  invalid_rows_mock1.csv
cleaned_input_mock2.csv  invalid_rows_mock2.csv


In [21]:
import pandas as pd

cleaned_df = pd.read_csv("data/processed/cleaned_input_mock2.csv")
invalid_df = pd.read_csv("data/processed/invalid_rows_mock2.csv")

print("Cleaned input:")
display(cleaned_df)

print("Invalid rows:")
display(invalid_df)

Cleaned input:


,source_id,source_type,raw_text,source_url,date,company_seed,source_type_cleaned,raw_text_cleaned,source_url_cleaned,date_cleaned,company_seed_cleaned,company_seed_invalid,company_seed_invalid_reason,row_invalid_reasons,row_valid_for_ner
0,1,SEC,Pfizer acquired Seagen in 2023.,https://example.com/sec1,2023-01,Pfizer,SEC,Pfizer acquired Seagen in 2023.,https://example.com/sec1,2023-01,Pfizer,False,NaN,NaN,True
1,2,SEC,Merck entered into a license agreement with Mo...,https://example.com/sec2,2024-02,Merck,SEC,Merck entered into a license agreement with Mo...,https://example.com/sec2,2024-02,Merck,False,NaN,NaN,True
2,3,USPTO,Regeneron and Sanofi jointly developed an anti...,https://example.com/uspto1,2022-06,Regeneron,USPTO,Regeneron and Sanofi jointly developed an anti...,https://example.com/uspto1,2022-06,Regeneron,False,NaN,NaN,True
3,4,PUBMED,CSL Behring collaborated with Takeda on plasma...,https://example.com/pubmed1,2021-11,CSL Behring,PUBMED,CSL Behring collaborated with Takeda on plasma...,https://example.com/pubmed1,2021-11,CSL Behring,False,NaN,NaN,True
4,5,SEC,Bristol Myers Squibb purchased Karuna Therapeu...,https://example.com/sec3,2024-12,Bristol Myers Squibb,SEC,Bristol Myers Squibb purchased Karuna Therapeu...,https://example.com/sec3,2024-12,Bristol Myers Squibb,False,NaN,NaN,True
5,6,USPTO,AbbVie filed patent with Genmab for a bispecif...,https://example.com/uspto2,2020-09,AbbVie,USPTO,AbbVie filed patent with Genmab for a bispecif...,https://example.com/uspto2,2020-09,AbbVie,False,NaN,NaN,True
6,7,SEC,AstraZeneca partnered with Daiichi Sankyo to c...,https://example.com/sec4,2023-08,AstraZeneca,SEC,AstraZeneca partnered with Daiichi Sankyo to c...,https://example.com/sec4,2023-08,AstraZeneca,False,NaN,NaN,True
7,8,PUBMED,Novartis worked with Alnylam on RNA interferen...,https://example.com/pubmed2,2022-03,Novartis,PUBMED,Novartis worked with Alnylam on RNA interferen...,https://example.com/pubmed2,2022-03,Novartis,False,NaN,NaN,True
8,9,SEC,10X GENOMICS INC acquired a small diagnostics ...,https://example.com/sec5,2024-03,10X Genomics,SEC,10X GENOMICS INC acquired a small diagnostics ...,https://example.com/sec5,2024-03,10X Genomics,False,NaN,NaN,True
9,10,SEC,10X GENOMICS BV entered into a collaboration w...,https://example.com/sec6,2024-04,10X Genomics,SEC,10X GENOMICS BV entered into a collaboration w...,https://example.com/sec6,2024-04,10X Genomics,False,NaN,NaN,True


Invalid rows:


,source_id,source_type,raw_text,source_url,date,company_seed,source_type_cleaned,raw_text_cleaned,source_url_cleaned,date_cleaned,company_seed_cleaned,company_seed_invalid,company_seed_invalid_reason,row_invalid_reasons,row_valid_for_ner
0,23,SEC,#NAME? filed a report.,https://example.com/bad1,2021-07,#NAME?,SEC,#NAME? filed a report.,https://example.com/bad1,2021-07,#NAME?,True,excel_error_token,excel_error_token,False
1,24,USPTO,11-JUL appears in the assignee field.,https://example.com/bad2,2020-05,11-JUL,USPTO,11-JUL appears in the assignee field.,https://example.com/bad2,2020-05,11-JUL,True,date_like_token,date_like_token,False
2,25,SEC,NaN,https://example.com/bad3,2022-02,Unknown,SEC,NaN,https://example.com/bad3,2022-02,Unknown,False,NaN,missing_raw_text,False
3,26,PUBMED,,https://example.com/bad4,2023-01,BlankCo,PUBMED,NaN,https://example.com/bad4,2023-01,BlankCo,False,NaN,missing_raw_text,False
4,27,SEC,N/A entered into a collaboration.,https://example.com/bad5,2022-06,NaN,SEC,N/A entered into a collaboration.,https://example.com/bad5,2022-06,NaN,True,missing_company_seed,missing_company_seed,False
5,28,USPTO,NULL filed patent with ???.,https://example.com/bad6,2021-09,NaN,USPTO,NULL filed patent with ???.,https://example.com/bad6,2021-09,NaN,True,missing_company_seed,missing_company_seed,False
6,50,UNKNOWN,Omega Therapeutics acquired ExampleCo.,https://example.com/unknown1,2022-07,Omega Therapeutics,UNKNOWN,Omega Therapeutics acquired ExampleCo.,https://example.com/unknown1,2022-07,Omega Therapeutics,False,NaN,unknown_source_type,False


# 4. Run spaCy + EntityRuler for NER

In [24]:
!python src/ner_spacy_entityruler.py \
  --input data/processed/cleaned_input_mock2.csv \
  --output outputs/ner/ner_results_mock2_entityruler.csv \
  --spacy-model en_core_web_sm

[ner_spacy_entityruler] Wrote 76 mention(s) to: outputs/ner/ner_results_mock2_entityruler.csv


In [25]:
ner_df = pd.read_csv("outputs/ner/ner_results_mock2_entityruler.csv")
display(ner_df)

,source_id,source_type,source_type_cleaned,source_url,source_url_cleaned,date,date_cleaned,company_seed,company_seed_cleaned,raw_text,raw_text_cleaned,raw_mention,entity_label,start_char,end_char,mention_source,mention_confidence
0,1,SEC,SEC,https://example.com/sec1,https://example.com/sec1,2023-01,2023-01,Pfizer,Pfizer,Pfizer acquired Seagen in 2023.,Pfizer acquired Seagen in 2023.,Pfizer,ORG,0,6,spacy_or_entityruler,NaN
1,1,SEC,SEC,https://example.com/sec1,https://example.com/sec1,2023-01,2023-01,Pfizer,Pfizer,Pfizer acquired Seagen in 2023.,Pfizer acquired Seagen in 2023.,Seagen,ORG,16,22,spacy_or_entityruler,NaN
2,2,SEC,SEC,https://example.com/sec2,https://example.com/sec2,2024-02,2024-02,Merck,Merck,Merck entered into a license agreement with Mo...,Merck entered into a license agreement with Mo...,Moderna,ORG,44,51,spacy_or_entityruler,NaN
3,3,USPTO,USPTO,https://example.com/uspto1,https://example.com/uspto1,2022-06,2022-06,Regeneron,Regeneron,Regeneron and Sanofi jointly developed an anti...,Regeneron and Sanofi jointly developed an anti...,Sanofi,ORG,14,20,spacy_or_entityruler,NaN
4,4,PUBMED,PUBMED,https://example.com/pubmed1,https://example.com/pubmed1,2021-11,2021-11,CSL Behring,CSL Behring,CSL Behring collaborated with Takeda on plasma...,CSL Behring collaborated with Takeda on plasma...,CSL Behring,ORG,0,11,spacy_or_entityruler,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71,58,SEC,SEC,https://example.com/sec24,https://example.com/sec24,2020-09,2020-09,Spark Therapeutics,Spark Therapeutics,University of Pennsylvania licensed technology...,University of Pennsylvania licensed technology...,Spark Therapeutics,ORG,50,68,spacy_or_entityruler,NaN
72,59,SEC,SEC,not_a_url,not_a_url,2024-05,2024-05,Alnylam,Alnylam,Alnylam acquired Example RNA Startup.,Alnylam acquired Example RNA Startup.,Alnylam,ORG,0,7,spacy_or_entityruler,NaN
73,59,SEC,SEC,not_a_url,not_a_url,2024-05,2024-05,Alnylam,Alnylam,Alnylam acquired Example RNA Startup.,Alnylam acquired Example RNA Startup.,Example RNA Startup,ORG,17,36,spacy_or_entityruler,NaN
74,60,PUBMED,PUBMED,https://example.com/pubmed14,https://example.com/pubmed14,NaN,NaN,Example Biotech,Example Biotech,Example Biotech worked with Example Diagnostics.,Example Biotech worked with Example Diagnostics.,Example Biotech,ORG,0,15,spacy_or_entityruler,NaN


# 5. Run alias resolution

In [26]:
!python src/resolve_alias.py \
  --input outputs/ner/ner_results_mock2_entityruler.csv \
  --resolved-output outputs/entity_resolution/resolved_entities_mock2_entityruler.csv \
  --review-output outputs/entity_resolution/review_queue_mock2_entityruler.csv

[resolve_alias] Wrote 76 resolved row(s) to: outputs/entity_resolution/resolved_entities_mock2_entityruler.csv
[resolve_alias] Wrote 4 review pair(s) to: outputs/entity_resolution/review_queue_mock2_entityruler.csv


In [27]:
import os

for path in [
    "outputs/entity_resolution/resolved_entities_mock2_entityruler.csv",
    "outputs/entity_resolution/review_queue_mock2_entityruler.csv",
]:
    print(path)
    print("exists:", os.path.exists(path))
    if os.path.exists(path):
        print("size:", os.path.getsize(path), "bytes")
    print("-" * 40)

outputs/entity_resolution/resolved_entities_mock2_entityruler.csv
exists: True
size: 17590 bytes
----------------------------------------
outputs/entity_resolution/review_queue_mock2_entityruler.csv
exists: True
size: 525 bytes
----------------------------------------


# 6. Candidate relations

In [28]:
import pandas as pd

resolved_df = pd.read_csv("outputs/entity_resolution/resolved_entities_mock2_entityruler.csv")
display(resolved_df)

,source_id,source_type,source_url,date,company_seed,raw_text,raw_mention,entity_label,start_char,end_char,normalized_name,base_name,legal_suffixes,canonical_group_id,canonical_name,merge_confidence,merge_decision,evidence_note
0,1,SEC,https://example.com/sec1,2023-01,Pfizer,Pfizer acquired Seagen in 2023.,Pfizer,ORG,0,6,PFIZER,PFIZER,NaN,17,Pfizer,0.88,auto_merge,auto_merge_component
1,1,SEC,https://example.com/sec1,2023-01,Pfizer,Pfizer acquired Seagen in 2023.,Seagen,ORG,16,22,SEAGEN,SEAGEN,NaN,36,Seagen,1.00,singleton,singleton_no_merge_needed
2,2,SEC,https://example.com/sec2,2024-02,Merck,Merck entered into a license agreement with Mo...,Moderna,ORG,44,51,MODERNA,MODERNA,NaN,22,Moderna,1.00,singleton,singleton_no_merge_needed
3,3,USPTO,https://example.com/uspto1,2022-06,Regeneron,Regeneron and Sanofi jointly developed an anti...,Sanofi,ORG,14,20,SANOFI,SANOFI,NaN,15,Sanofi,1.00,singleton,singleton_no_merge_needed
4,4,PUBMED,https://example.com/pubmed1,2021-11,CSL Behring,CSL Behring collaborated with Takeda on plasma...,CSL Behring,ORG,0,11,CSL BEHRING,CSL BEHRING,NaN,21,CSL Behring,0.88,auto_merge,auto_merge_component
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71,58,SEC,https://example.com/sec24,2020-09,Spark Therapeutics,University of Pennsylvania licensed technology...,Spark Therapeutics,ORG,50,68,SPARK THERAPEUTICS,SPARK THERAPEUTICS,NaN,1,Spark Therapeutics,0.50,review_needed,high_string_similarity
72,59,SEC,not_a_url,2024-05,Alnylam,Alnylam acquired Example RNA Startup.,Alnylam,ORG,0,7,ALNYLAM,ALNYLAM,NaN,35,Alnylam,1.00,singleton,singleton_no_merge_needed
73,59,SEC,not_a_url,2024-05,Alnylam,Alnylam acquired Example RNA Startup.,Example RNA Startup,ORG,17,36,EXAMPLE RNA STARTUP,EXAMPLE RNA STARTUP,NaN,31,Example RNA Startup,1.00,singleton,singleton_no_merge_needed
74,60,PUBMED,https://example.com/pubmed14,NaN,Example Biotech,Example Biotech worked with Example Diagnostics.,Example Biotech,ORG,0,15,EXAMPLE BIOTECH,EXAMPLE BIOTECH,NaN,6,Example Biotech,1.00,singleton,singleton_no_merge_needed


In [29]:
!mkdir -p outputs/relations

!python src/extract_candidate_relations.py \
  --input outputs/entity_resolution/resolved_entities_mock2_entityruler.csv \
  --output outputs/relations/candidate_relations_mock2_entityruler.csv

[extract_candidate_relations] Wrote 22 candidate row(s) to: outputs/relations/candidate_relations_mock2_entityruler.csv
